# 08 · Scoring batch de l'attrition (modèle @champion) + segmentation
Le Feature Store retrouve les features à partir des seules clés (pas d'écart entraînement/production).
Écriture idempotente : relancer le scoring d'un mois remplace ce mois, sans doublon.

In [0]:
%pip install -q databricks-feature-engineering
%restart_python

In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))
import numpy as np
import mlflow
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient
from churn.dbx import get_config
from churn.config import KEYS

cfg = get_config(dbutils)
fe = FeatureEngineeringClient()
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

ft = spark.table(cfg.t("client_features"))
snapshot = ft.agg(F.max("snapshot_date")).first()[0]                  # la date la plus récente
keys = ft.where(F.col("snapshot_date") == snapshot).select(*KEYS)    # uniquement les clés
champion = client.get_model_version_by_alias(cfg.churn_model_name, "champion")
print(f"Scoring du {snapshot} : {keys.count()} clients, modèle version {champion.version}")

In [0]:
# 1) Probabilité d'attrition : le Feature Store ajoute les features, le champion prédit
scored = (fe.score_batch(model_uri=f"models:/{cfg.churn_model_name}@champion", df=keys,
                         result_type="double")
          .withColumnRenamed("prediction", "p_churn"))
pdf = scored.toPandas()

# 2) Segment comportemental, avec le modèle de segmentation champion
seg_model = mlflow.pyfunc.load_model(f"models:/{cfg.segment_model_name}@champion")
pdf["segment_comportemental"] = np.asarray(seg_model.predict(pdf)).ravel()

# 3) Classe de risque lisible par un conseiller (taux de base ~3 %)
pdf["risk_band"] = pdf.p_churn.map(lambda p: "élevé" if p >= 0.20 else "moyen" if p >= 0.05 else "faible")
pdf["model_version"] = int(champion.version)

# 4) Écriture idempotente : on remplace uniquement le mois scoré
out = spark.createDataFrame(pdf).withColumn("scored_at", F.current_timestamp())
table = cfg.t("attrition_predictions")
if not spark.catalog.tableExists(table):
    out.write.saveAsTable(table)
else:
    (out.write.mode("overwrite").option("replaceWhere", f"snapshot_date = '{snapshot}'")
     .option("mergeSchema", "true").saveAsTable(table))
print("Écrit dans", table)

In [0]:
display(spark.table(cfg.t("attrition_predictions"))
        .where(F.col("snapshot_date") == snapshot)
        .groupBy("segment_comportemental", "risk_band")
        .agg(F.count("*").alias("clients"), F.round(F.avg("p_churn"), 3).alias("p_moyenne"))
        .orderBy("segment_comportemental", "risk_band"))

In [0]:
display(spark.table(cfg.t("attrition_predictions"))
        .where(F.col("snapshot_date") == snapshot)
        .orderBy(F.desc("p_churn"))
        .select("client_id", F.round("p_churn", 3).alias("p_churn"), "risk_band", "segment_comportemental",
                "n_salary_90d", "ext_transfer_out_90d", "n_complaints_90d", "fees_90d",
                "app_trend_30_90", "has_credit_immo")
        .limit(10))